# Product Recognizer GTIN using Gemini

<a href="https://colab.research.google.com/github/cjk0604/Gemini_product_recognizer_gtin/blob/main/product_recognizer_gtin_using_Gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab">
</a>
<a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fcjk0604%2FGemini_product_recognizer_gtin%2Fmain%2Fproduct_recognizer_gtin_using_Gemini.ipynb">
  <img src="https://img.shields.io/badge/Colab_Enterprise-Open-blue?logo=google-cloud" alt="Open In Colab Enterprise">
</a>
<a href="https://github.com/cjk0604/Gemini_product_recognizer_gtin/blob/main/product_recognizer_gtin_using_Gemini.ipynb">
  <img src="https://img.shields.io/badge/GitHub-View_Source-black?logo=github" alt="View on GitHub">
</a>


In [ ]:
#@title Imports
# Install libraries that are not installed in the default runtime.
!pip install retrying
!pip install pandarallel

from IPython import display
import PIL
import PIL.ImageDraw
import PIL.ImageFont
from collections.abc import Mapping, Sequence
import dataclasses
import io
import ipywidgets
import json
import logging
import multiprocessing
import os
import pandas as pd
import pandarallel
import random
import requests
import retrying
from rich import print
import time
import tqdm
from typing import Any

from google.cloud import aiplatform
from google.cloud import storage

from vertexai import vision_models
from google import genai
from google.genai.types import (
    FunctionDeclaration,
    GenerateContentConfig,
    GoogleSearch,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
    ThinkingConfig,
    Tool,
    ToolCodeExecution,
)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 4.0 MB/s eta 0:00:00
  Created wheel for pandarallel: filename=pandarallel-1.6.5-py3-none-any.whl size=16676 sha256=722a06289dff9e4301335100891f89f89bf1927c4d961bbe0bfd37e1c638026c
  Stored in directory: /root/.cache/pip/wheels/50/4f/1e/34e057bb868842209f1623f195b74fd7eda229308a7352d47f
Successfully built pandarallel


/usr/local/lib/python3.10/dist-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
#@title Global Parameters & Initialization

EMBEDDING_DIMENSION = 128  # @param {type: "integer"}
PROJECT_ID = ""  # @param {type: "string"}
LOCATION = "us-central1"  # @param {type: "string"}
BUCKET = ""  # @param {type: "string"}

# Custom csv dataset
DATASET = ""  # @param {type: "string"}

# Jsonl files containing all the queries and ground truth.
QUERYSET = ""  # @param {type: "string"}
CACHE_DIR = "cache"  # @param {type: "string"}
DEPLOYED_INDEX_ID = ""  # @param {type: "string"}
NUM_NEIGHBORS = 20  # @param {type: "integer"}
# @markdown See available options
# @markdown [here](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions#embeddings_stable_model_versions).
EMBEDDING_MODEL_NAME = ""  # @param {type: "string"}
# @markdown See available options
# @markdown [here](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions#stable-versions-available).
GEMINI_MODEL_VERSION = "gemini-2.0-flash-001"  # @param {type: "string"}

RETRY_STRATEGY = {
    "stop_max_attempt_number": 5,
    "wait_exponential_multiplier": 1000,  # 1s
    "wait_exponential_max": 10000,  # 10s
    "retry_on_exception": lambda e: (
        "403 Client Error" not in str(e)  # Does not exist.
        and not isinstance(e, requests.exceptions.MissingSchema) # Invalid URI.
    ),
}

# Initialize Vertex AI.
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Limit logs from the Matching Engine SDK since we are already tracking progress
# using tqdm and using parallelism.
logging.getLogger(
    'google.cloud.aiplatform.matching_engine.matching_engine_index'
).setLevel(logging.ERROR)


In [ ]:
from collections import defaultdict

#@title [Library] Vertex Matching Engine
# The embedding model.
EMBEDDING_MODEL = vision_models.MultiModalEmbeddingModel.from_pretrained(
    EMBEDDING_MODEL_NAME
)

# The GCS bucket that will be accessed.
GCS_CLIENT = storage.Client()
GCS_BUCKET = GCS_CLIENT.get_bucket(BUCKET)

GCS_PREFIX = "gs://"
SUPPORTED_EXTENSIONS = ("jpg", "jpeg", "png", "gif")


@dataclasses.dataclass(frozen=True)
class GcsUri:
  bucket_name: str
  object_name: str

  def from_parts(bucket_name: str, object_name: str) -> "GcsUri":
    ext = object_name.split(".")[-1]
    if ext not in SUPPORTED_EXTENSIONS:
      raise ValueError(
          f"Invalid object extension {ext}"
          f"(supported extensions: {SUPPORTED_EXTENSIONS})"
      )
    return GcsUri(bucket_name=bucket_name, object_name=object_name)

  def from_path(gcs_path: str) -> "GcsUri":
    """Constructs a GcsUri from a gcs path."""
    if not gcs_path.startswith(GCS_PREFIX):
      raise ValueError(f"Gcs Path must begin with '{GCS_PREFIX}'.")
    if "/" not in gcs_path:
      raise ValueError(f"Gcs Path must contain an object.")
    return GcsUri.from_parts(*gcs_path.removeprefix(GCS_PREFIX).split("/", 1))

  @retrying.retry(**RETRY_STRATEGY)
  def from_web_uri(web_uri: str, skip_download: bool=False) -> "GcsUri":
    """Constructs a GcsUri from an image at a web uri."""
    # Construct the path to the GCS copy of the image.
    cache_path = os.path.join(CACHE_DIR, "images", web_uri)
    if cache_path.split(".")[-1] not in SUPPORTED_EXTENSIONS:
      cache_path += ".jpeg"
    blob = GCS_BUCKET.blob(cache_path)

    # If the image is not already cached to GCS, we cache it.
    if not blob.exists():
      if not skip_download:
        response = requests.get(web_uri)
        response.raise_for_status()
        blob.upload_from_string(response.content)
      else:
        return None
    return GcsUri.from_parts(bucket_name=BUCKET, object_name=cache_path)

  def to_path(self) -> str:
    """Converts a GcsUri into a gcs path."""
    return f"gs://{self.bucket_name}/{self.object_name}"

  def to_part(self) -> Part:
    """Converts a GCS URI into a generative model Part."""
    return Part.from_uri(file_uri=self.to_path(), mime_type=self.get_mime_type())

  def to_bytes(self) -> bytes:
    """Converts a GCS Uri into bytes."""
    image_blob = GCS_BUCKET.blob(self.object_name)
    return image_blob.download_as_bytes()

  def to_image(self) -> PIL.Image:
    """Constructs a Pillow Image from the GCS Uri."""
    return PIL.Image.open(io.BytesIO(self.to_bytes()))

  def get_mime_type(self) -> str:
    """Gets the mime type of the object."""
    if self.object_name.endswith(".png"):
      return "image/png"
    elif self.object_name.endswith(".gif"):
      return "image/gif"
    else:
      return "image/jpeg"

  @retrying.retry(**RETRY_STRATEGY)
  def get_image_embedding(self, allow_cache: bool = True) -> Sequence[float]:
    """Gets an image embedding vector."""
    # If the embedding is present in the cache, we load it to avoid recomputation.
    cache_path = os.path.join(CACHE_DIR, "image_embeddings", self.object_name)
    blob = GCS_BUCKET.blob(cache_path)
    if blob.exists() and allow_cache:
      embedding_data = blob.download_as_string().decode().split(',')
      image_embedding = [float(d) for d in embedding_data]
    else:
      image = vision_models.Image(
          gcs_uri=f"gs://{self.bucket_name}/{self.object_name}"
      )
      embeddings_response = EMBEDDING_MODEL.get_embeddings(
          image=image,
          dimension=EMBEDDING_DIMENSION,
      )
      image_embedding = embeddings_response.image_embedding
      if allow_cache:
        embedding_data = ",".join(str(d) for d in image_embedding)
        blob.upload_from_string(embedding_data.encode())
    return image_embedding

  @retrying.retry(**RETRY_STRATEGY)
  def get_image_text_embedding(self, text_context: str, allow_cache: bool = True) -> Sequence[float]:
    """Gets an image embedding vector."""
    # If the embedding is present in the cache, we load it to avoid recomputation.
    cache_path = os.path.join(CACHE_DIR, "image_text_embeddings", self.object_name)
    blob = GCS_BUCKET.blob(cache_path)
    if blob.exists() and allow_cache:
      embedding_data = blob.download_as_string().decode().split(',')
      image_embedding = [float(d) for d in embedding_data]
    else:
      image = vision_models.Image(
          gcs_uri=f"gs://{self.bucket_name}/{self.object_name}"
      )
      embeddings_response = EMBEDDING_MODEL.get_embeddings(
          image=image,
          contextual_text=text_context,
          dimension=EMBEDDING_DIMENSION,
      )
      image_embedding = embeddings_response.image_embedding
      if allow_cache:
        embedding_data = ",".join(str(d) for d in image_embedding)
        blob.upload_from_string(embedding_data.encode())
    return image_embedding


  @retrying.retry(**RETRY_STRATEGY)
  def find_neighbors(
      self, image_embedding: Sequence[float] | None = None,
  ) -> Sequence["Match"]:
    """Finds the nearest neighbors of the image."""
    image_embedding = image_embedding or self.get_image_embedding()
    matches = []
    for match_neighbor in INDEX_ENDPOINT.find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[image_embedding],
        return_full_datapoint=True,
        num_neighbors=NUM_NEIGHBORS,
    )[0]:
      metadata = json.loads("".join(match_neighbor.restricts[0].allow_tokens))
      metadata["image"] = GcsUri.from_path(metadata["image"])
      matches.append(
          Match(score=match_neighbor.distance, product=Product(**metadata))
      )
    return matches


  @retrying.retry(**RETRY_STRATEGY)
  def find_neighbors_with_dedup(
      self, image_embedding: Sequence[float] | None = None,
  ) -> Sequence["Match"]:
    """Finds the nearest neighbors of the image."""
    image_embedding = image_embedding or self.get_image_embedding()
    matches = []
    gtin_set = set()
    for match_neighbor in INDEX_ENDPOINT.find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[image_embedding],
        return_full_datapoint=True,
        num_neighbors=NUM_NEIGHBORS,
    )[0]:
      metadata = json.loads("".join(match_neighbor.restricts[0].allow_tokens))
      metadata["image"] = GcsUri.from_path(metadata["image"])
      for gtin in metadata["gtins"]:
        if gtin not in gtin_set:
          matches.append(
            Match(score=match_neighbor.distance, product=Product(**metadata))
            )
          break
      gtin_set.update(metadata["gtins"])
    return matches


  @retrying.retry(**RETRY_STRATEGY)
  def find_neighbors_with_dedup2(
      self, image_embedding: Sequence[float] | None = None,
  ) -> Sequence["Match"]:
    """Finds the nearest neighbors of the image, but deduping using image path
      Something is still wrong.
    """
    image_embedding = image_embedding or self.get_image_embedding()
    matches = []
    gtin_set = defaultdict(int)
    for match_neighbor in INDEX_ENDPOINT.find_neighbors(
        deployed_index_id=DEPLOYED_INDEX_ID,
        queries=[image_embedding],
        return_full_datapoint=True,
        num_neighbors=NUM_NEIGHBORS,
    )[0]:
      metadata = json.loads("".join(match_neighbor.restricts[0].allow_tokens))
      image_path = metadata["image"]
      metadata["image"] = GcsUri.from_path(image_path)
      for gtin in metadata["gtins"]:
        if gtin_set[gtin] <= 1:
          matches.append(
            Match(score=match_neighbor.distance, product=Product(**metadata))
            )
          gtin_set[gtin] += 1
          break

    return matches


@dataclasses.dataclass(frozen=True)
class Product:
  entity_uuid: str
  product_title: str
  gtins: list[int]
  image: GcsUri
  brand_name: str | None = None

  def from_df_row(
      row: tuple[int, pd.core.series.Series] | pd.core.series.Series,
  ) -> "Product":
    """Constructs a Product from a pandas Series."""
    if isinstance(row, tuple):
      row = row[1]
    return Product(**{name: value for name, value in zip(row.index, row)})

  def get_datapoint_id(self) -> str:
    return self.image.object_name.replace("/", "_")

  def to_datapoint(self) -> Mapping[str, Any]:
    """Converts the Product into a Datapoint for Vertex Matching Engine."""
    # Note that there is no metadata field for Vertex Matching Engine. Thus, we
    # elect to store metadata in the "restricts[0].allow_list" by convention.
    return {
          "datapoint_id": self.get_datapoint_id(),
          "feature_vector": self.image.get_image_embedding(),
          "restricts": [{
              "namespace": "metadata",
              "allow_list": json.dumps({
                  "gtins": self.gtins,
                  "image": self.image.to_path(),
                  "product_title": self.product_title,
                  "entity_uuid": self.entity_uuid,
                  "brand_name": self.brand_name,
              }),
          }],
      }

  def to_datapoint2(self) -> Mapping[str, Any]:
    """Converts the Product into a Datapoint for Vertex Matching Engine."""
    # Note that there is no metadata field for Vertex Matching Engine. Thus, we
    # elect to store metadata in the "restricts[0].allow_list" by convention.
    return {
          "datapoint_id": self.get_datapoint_id(),
          "feature_vector": self.image.get_image_text_embedding(
              self.product_title),
          "restricts": [{
              "namespace": "metadata",
              "allow_list": json.dumps({
                  "gtins": self.gtins,
                  "image": self.image.to_path(),
                  "product_title": self.product_title,
                  "entity_uuid": self.entity_uuid,
                  "brand_name": self.brand_name,
              }),
          }],
    }

  @retrying.retry(**RETRY_STRATEGY)
  def get_visualization(self, height: float = 200) -> PIL.Image:
    """Gets a visualization of this Product."""
    # Load the image and scale it to a height of 200 maintaining aspect ratio.
    image = self.image.to_image()
    width, height = image.size
    width, height = width * 200 // height, 200
    image = image.resize((width, height), PIL.Image.BILINEAR)

    # Create a drawing.
    draw = PIL.ImageDraw.Draw(image)

    # Add the title to the top.
    #if self.product_title is not None:
    #  font_size = width * 2 / len(self.product_title)
    #  font = PIL.ImageFont.load_default(size=font_size)
    #  draw.text((0, 0), self.product_title, (0, 0, 0), font=font)

    # Add the GTIN to the bottom right.
    #gtin = str(self.gtins[0])
    #font_size = width / len(gtin)
    #font = PIL.ImageFont.load_default(size=font_size)
    #draw.text((width / 4, height - font_size - 10), gtin, (0, 0, 0), font=font)

    return image


@dataclasses.dataclass
class Match:
  score: float
  product: Product



# @markdown The dataset is a CSV file within the `GCS_BUCKET` following this
# @markdown header:
# @markdown ```
# @markdown entity_uuid,product_title,brand_name,gtins,images
# @markdown ...
# @markdown ```
# @markdown
# @markdown The queryset is a JSONL file where each row contains a JSON with the
# @markdown "imageUri" key.

BATCH_SIZE = 100


def load_dataset(
    dataset: str, nb_workers: int = 12, limit: int | None = None
) -> Sequence[Product]:
  """Loads a CSV dataset into a sequence of Products.
     required columns: gtins, images
  """

  def from_web_uri_no_exception(web_uri: str) -> GcsUri | None:
    try:
      return GcsUri.from_web_uri(web_uri)
    except Exception as e:
      return

  pandarallel.pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)

  # Extract the dataset CSV text.
  blob = GCS_BUCKET.blob(dataset)
  text = blob.download_as_string().decode("utf-8")

  # Load the CSV text into a dataframe for easier processing.
  df = pd.read_csv(io.StringIO(text), header=0)

  # Extract each of the images into its own row named "image".
  df["gtins"] = df["gtins"].apply(lambda gtins: gtins.split('|'))
  df["images"] = df["images"].apply(lambda images: images.split('||'))
  df = df.explode("images")
  df.rename(columns={"images": "image"}, inplace=True)
  if limit is not None:
    df = df.head(limit)

  # Load into the dataframe.
  print("Saving images to GCS...")
  df["image"] = df["image"].parallel_apply(from_web_uri_no_exception)
  df = df.dropna()
  print(f"Total number of rows: {len(df)}")
  return [Product.from_df_row(row) for row in df.iterrows()]


def load_queryset(
    queryset: str, processes: int = 12, limit: int | None = None,
) -> Sequence[GcsUri]:
  """Loads a JSONL queryset into a sequence of GcsUris.
     required field: imageUri
  """

  # Download the jsonl and parse the rows.
  blob = GCS_BUCKET.blob(queryset)
  text = blob.download_as_string().decode("utf-8")
  rows = [json.loads(row) for row in text.split("\n") if row.strip()]

  image_uris = [row["imageUri"] for row in rows]
  if limit is not None:
    image_uris = random.choices(image_uris, k=limit)

  with multiprocessing.Pool(processes=processes) as pool:
    return list(tqdm.tqdm(
        pool.imap(GcsUri.from_web_uri, image_uris),
        total=len(image_uris),
    ))


def upsert_datapoints(products: Sequence[Product]) -> int:
  datapoints=[product.to_datapoint() for product in products]
  start_time = time.perf_counter()
  INDEX.upsert_datapoints(datapoints=datapoints)
  end_time = time.perf_counter()
  return end_time - start_time


def upsert_dataset(products: Sequence[Product]) -> Sequence[int]:
  """Loads Products into an index."""
  batches = [
      products[i:i + BATCH_SIZE] for i in range(0, len(products), BATCH_SIZE)
  ]
  with multiprocessing.Pool(processes=6) as pool:
    return list(tqdm.tqdm(
        pool.imap(upsert_datapoints, batches),
        total=len(batches),
    ))

def upsert_datapoints2(products: Sequence[Product]) -> int:
  """Version 2, add text into the embedding using product title."""
  datapoints=[product.to_datapoint2() for product in products]
  start_time = time.perf_counter()
  INDEX.upsert_datapoints(datapoints=datapoints)
  end_time = time.perf_counter()
  return end_time - start_time


def upsert_dataset2(products: Sequence[Product]) -> Sequence[int]:
  """Loads Products into an index."""
  batches = [
      products[i:i + BATCH_SIZE] for i in range(0, len(products), BATCH_SIZE)
  ]
  with multiprocessing.Pool(processes=6) as pool:
    return list(tqdm.tqdm(
        pool.imap(upsert_datapoints2, batches),
        total=len(batches),
    ))


@retrying.retry(**RETRY_STRATEGY)
def visualize_products(
    products: Sequence[Match] | Sequence[Product] | Product | GcsUri,
    vis_height: int=200
) -> None:
  """Visualizes one or multiple products."""
  if isinstance(products, Product):
    image = products.get_visualization()
    image_buffer = io.BytesIO()
    image.save(image_buffer, format="jpeg")
    display.display(ipywidgets.Image(value=image_buffer.getvalue()))
    return
  elif isinstance(products, GcsUri):
    image = products.to_image()
    width, height = image.size
    width, height = width * vis_height // height, vis_height
    image = image.resize((width, height), PIL.Image.BILINEAR)
    image_buffer = io.BytesIO()
    image.save(image_buffer, format="jpeg")
    display.display(ipywidgets.Image(value=image_buffer.getvalue()))
    return
  elif not products:
    return
  elif isinstance(products[0], Match):
    products = [result.product for result in products]
  images = [product.get_visualization() for product in products]
  buffer = 10
  width = sum(image.width for image in images) + buffer * (len(images) - 1)
  height = max(image.height for image in images)
  output_image = PIL.Image.new("RGB", (width, height), (255, 255, 255))

  x_offset = 0
  for image in images:
    output_image.paste(image, (x_offset, 0))
    x_offset += image.width + buffer

  image_buffer = io.BytesIO()
  output_image.save(image_buffer, format="jpeg")
  display.display(ipywidgets.Image(value=image_buffer.getvalue()))


def save_string_to_gcs(gcs_url: str, file_content: str) -> None:
  storage_client = storage.Client()
  bucket_name, blob_name = gcs_url.replace("gs://", "").split("/", 1)
  bucket = storage_client.bucket(bucket_name)
  blob = bucket.blob(blob_name)
  blob.upload_from_string(file_content)

In [ ]:
#@title [Library] Gemini Reranking
import copy
import dataclasses
from google.genai import types


LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

if "2.5" in GEMINI_MODEL_VERSION:
  #thinking_config=ThinkingConfig(thinking_budget=0)
  thinking_config=ThinkingConfig()
else:
  thinking_config=None

gemini_config=GenerateContentConfig(
        temperature=0.0,
        top_p=0.95,
        candidate_count=1,
        thinking_config=thinking_config,
        system_instruction="""
    You are a product search engine and your responsibility is to recognize
    the product in the query image(s) by selecting the closest product from
    a set of candidates based on their images and/or product titles. Note that a
    query might have multiple images but they belong to the same product.

    Please follow these instructions:

    a. The overall picture layout and content of the candidate should match exactly with the product in the query image.
    b. The title of the candidate should also match the product in the query image.
    c. Pay attention to details on the product label like product name, product variant,
        product weight, size and quantity.
    d. You should also pay attention to the package type, shape and color scheme which could
        indicate different variants, weight or quantity.
    e. The query images are captured in non-ideal conditions
        and might have distortions or occlusions. If the image is blurry or not complete, you
        should find the best match based on partial similarity of graphics and text layout on the packaging.

    First you should output the product's title based on the query image, including product
    brand, variant and weight/quantity. Then output the index of the closest matching
    candidate based on their images and titles.

    Please output in a json format like this:
    {
      "title" : <product title>
      "candidate": <index of the closest match>
    }

    """,
    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)

gemini_punting_config = gemini_config=GenerateContentConfig(
        temperature=0.0,
        top_p=0.95,
        candidate_count=1,
        thinking_config=thinking_config,
        system_instruction="""
    You are a product search engine and your responsibility is to recognize
    the product in the query image(s) by selecting the closest product from
    a set of candidates based on their images and/or product titles. Note that a
    query might have multiple images but they belong to the same product.

    Please follow these instructions:

    a. The overall picture layout and content of the candidate should match exactly with the product in the query image.
    b. The title of the candidate should also match the product in the query image.
    c. Pay attention to details on the product label like product name, product variant,
        product weight, size and quantity.
    d. You should also pay attention to the package type, shape and color scheme which could
        indicate different variants, weight or quantity.
    e. The query images are captured in non-ideal conditions
        and might have distortions or occlusions. If the image is blurry or occluded, you
        should find the best match based on partial similarity of graphics and text layout on the packaging.
    f. You should prioritize the use of the original query image to find a match, and
      only rely on the zoomed out query image to add evidence from its neighboring product.

    First you should output the product's title based on the query image, including product
    brand, variant and weight/quantity. Then output the index of the closest matching
    candidate based on their images and titles.

    If you are sure about the following cases:
    a. The first query image contains no product -- the shelf is empty.
    b. The first query image is a price tag mis-detected as a product.
    c. No matching product can be found in the query images.
    d. The best matching product has a different brand than the query product.
    You should set the index of the candidate to -1.

    Please output in a json format like this:
    {
      "color": <the colors on the package of the query product>
      "title" : <title of the query product>
      "candidate": <index of the closest match>
      "reason" : <The reason of the match>
    }
    """,
    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)


gemini_punting_letter_config = gemini_config=GenerateContentConfig(
        temperature=0.0,
        top_p=0.95,
        candidate_count=1,
        thinking_config=thinking_config,
        system_instruction="""
    You are a product search engine and your responsibility is to recognize
    the product in the query image(s) by selecting the closest product from
    a set of candidates based on their images and/or product titles. Note that a
    query might have multiple images but they belong to the same product.

    Please follow these instructions:

    a. The overall picture layout and content of the candidate should match exactly with the product in the query image.
    b. The title of the candidate should also match the product in the query image.
    c. Pay attention to details on the product label like product name, product variant,
        product weight, size and quantity.
    d. You should also pay attention to the package type, shape and color scheme which could
        indicate different variants, weight or quantity.
    e. The query images are captured in non-ideal conditions
        and might have distortions or occlusions. If the image is blurry or occluded, you
        should find the best match based on partial similarity of graphics and text layout on the packaging.
    f. You should prioritize the use of the original query image to find a match, and
      only rely on the zoomed out query image to add evidence from its neighboring product.

    First you should output the product's title based on the query image, including product
    brand, variant and weight/quantity. Then output the index letter of the closest matching
    candidate based on their images and titles.

    If you are sure about the following cases:
    a. The first query image contains no product -- the shelf is empty.
    b. The first query image is a price tag mis-detected as a product.
    c. No matching product can be found in the query images.
    d. The best matching product has a different brand than the query product.
    You should set the index letter of the candidate to '0'.

    Please output in a json format like this:
    {
      "color": <the colors on the package of the query product>
      "title" : <title of the query product>
      "candidate": <index letter of the closest match>
    }
    """,
    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)

gemini_punting_letter2_config = gemini_config=GenerateContentConfig(
        temperature=0.0,
        top_p=0.95,
        candidate_count=1,
        thinking_config=thinking_config,
        system_instruction="""
    You are a product search engine and your responsibility is to recognize
    the product in the query image(s) by selecting the closest product from
    a set of candidates based on their images and/or product titles. Note that a
    query might have multiple images but they belong to the same product.

    For each candidate product, imagine the 3D shape of the product and how it
    will appear when placed on the shelf like the product in the query image, and
    then compare it with the actual query image to identify similarities and differences.
    Please follow also these instructions:

    a. The overall picture layout and content of the candidate should match exactly with the product in the query image.
    b. The title of the candidate should also match the product in the query image.
    c. Pay attention to details on the product label like product name, product variant,
        product weight, size and quantity.
    d. You should also pay attention to the package type, shape and color scheme which could
        indicate different variants, weight or quantity.
    e. The query images are captured in non-ideal conditions
        and might have distortions or occlusions. If the image is blurry or occluded, you
        should find the best match based on partial similarity of graphics and text layout on the packaging.
    f. You should prioritize the use of the original query image to find a match, and
      only rely on the zoomed out query image to add evidence from its neighboring product.

    First you should output the product's title based on the query image, including product
    brand, variant and weight/quantity. Then output the index letter of the closest matching
    candidate based on their images and titles.

    If you are sure about the following cases:
    a. The first query image contains no product -- the shelf is empty.
    b. The first query image is a price tag mis-detected as a product.
    c. No matching product can be found in the query images.
    d. The best matching product has a different brand than the query product.
    You should set the index letter of the candidate to '0'.

    Please output in a json format like this:
    {
      "color": <the colors on the package of the query product>
      "title" : <title of the query product>
      "candidate": <index letter of the closest match>
    }
    """,
    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)


cot_w_punting_config = gemini_config=GenerateContentConfig(
        temperature=0.0,
        top_p=0.95,
        candidate_count=1,
        thinking_config=thinking_config,
        system_instruction="""
    You are a product search specialist tasked to find the exact matching product
    from a given query image.
    Please follow these instructions:

    a. A query might have multiple images but they all belong to the same product.
    b. You should select the best matching from the list of candidate products.
    c. A candidate matches the query image if all of the following is true:
       1. Exactly the same overall packaging layout and content as in the query image.
       2. The title or brand of the candidate should match the query image.
       3. If visible, product variant, weight, size and quantity should match the query image.
       4. Package type, shape and color scheme should match the query image.
    d. The query images are captured in non-ideal conditions
        and might have distortions or occlusions. If the image is blurry or occluded, you
        should find the best match based on partial similarity of graphics and text layout on the packaging.
    e. You should prioritize the use of the original query image to find a match, and
      only rely on the zoomed out query image to add evidence from its neighboring product.

    First you should output the product's title and color based on the query image, including product
    brand, variant and weight/quantity. Then output the index of the closest matching
    candidate based on their images and titles.

    If you are sure about the following cases:
    a. The first query image contains no product -- the shelf is empty.
    b. The first query image is a price tag mis-detected as a product.
    c. No matching product can be found in the query images.
    d. The best matching product has a different brand than the query product.
    You should set the index of the candidate to -1.

    Please output in a json format like this:
    {
      "color": <the colors on the package of the query product>
      "title" : <title of the query product>
      "candidate": <index of the closest match>
    }
    """,
    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)


cot2_w_punting_config = gemini_config=GenerateContentConfig(
        temperature=0.0,
        top_p=0.95,
        candidate_count=1,
        thinking_config=thinking_config,
        system_instruction="""
    You are a product search specialist tasked to find the exact matching product
    from a given query image.
    Please follow these instructions:

    a. a query might have multiple images but they belong to the same product.
    b. You should select the best matching from the list of candidate products based on their title and image.
    c. A product does not match the query image if any the following is true:
       1. The title or brand of the candidate does match the query image.
       2. If visible, product variant, weight, size and quantity does not match the query image.
       3. Package type, shape and color layout does not match the query image.
    d. The query images are captured in non-ideal conditions
        and might have distortions or occlusions. If the image is blurry or occluded, you
        should find the best match based on partial similarity of graphics and text layout on the packaging.
    e. You should prioritize the use of the original query image to find a match, and
      only rely on the zoomed out query image to add evidence from its neighboring product.

    First you should output the product's title based on the query image, including product
    brand, variant and weight/quantity. Then output the index of the closest matching
    candidate based on their images and titles.

    If you are sure about the following cases:
    a. The first query image contains no product -- the shelf is empty.
    b. The first query image is a price tag mis-detected as a product.
    c. No matching product can be found in the query images.
    d. The best matching product has a different brand than the query product.
    You should set the index of the candidate to -1.

    Please output in a json format like this:
    {
      "color": <the colors on the package of the query product>
      "title" : <title of the query product>
      "candidate": <index of the closest match>
      "reason" : <The reason of the match>
    }
    """,
    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)


gemini_title_config=GenerateContentConfig(
        temperature=0.0,
        top_p=0.95,
        candidate_count=1,
        thinking_config=thinking_config,
        system_instruction="""
    You are a product specialist for an online merchant.

    Please write a product title for the product in the center of the query image,
    the title should include brand, product name, product variant, weight/size/quantity.

    Please only output the title as text and nothing else.
    """,
        media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)


from typing import Union

def create_product_recognition_prompt_single(
    query_image: GcsUri,
    matches: Sequence[Match],
) -> Sequence[Any] | None:
  """Creates the Product Recognition Gemini reranking prompt."""
  if query_image.object_name is None or not matches:
    return None

  prompt = [
    """<input>"""
  ]
  prompt.extend(["Query image: ", query_image.to_part(), ";\n"])

  for i, result in enumerate(matches):
    prompt.append(f"Candidate {i} : ")
    prompt.append(f"title - {result.product.product_title}, ")
    prompt.extend(["image - ", result.product.image.to_part(), ";\n"])
  prompt.append("</input>")
  return prompt


def create_product_recognition_prompt_margin_debug(
    query_image: GcsUri,
    query_image_with_margin: Union[GcsUri, None],
    matches: Sequence[Match],
    candidate_index_start: int = 0
) -> Sequence[Any] | None:
  """Creates the Product Recognition Gemini reranking prompt."""
  if query_image.object_name is None or not matches:
    return None

  prompt = ["<input>"]
  if query_image_with_margin is None:
    prompt.extend(["Query image: ", query_image.to_part(), ";\n"])
  else:
    prompt.extend(["Query image: ", query_image.to_part(), ", Zoomed out image: ",
                   query_image_with_margin.to_part(), ";\n"])

  for i, result in enumerate(matches):
    prompt.append(f"Candidate {i + candidate_index_start} : ")
    prompt.append(f"title - {result.product.product_title}, ")
    prompt.extend(["image - ", result.product.image.to_part(), ";\n"])
  prompt.append("</input>")
  return prompt


def create_product_recognition_prompt_margin_letter(
    query_image: GcsUri,
    query_image_with_margin: Union[GcsUri, None],
    matches: Sequence[Match],
) -> Sequence[Any] | None:
  """Creates the Product Recognition Gemini reranking prompt."""
  if query_image.object_name is None or not matches:
    return None

  prompt = ["<input>"]
  if query_image_with_margin is None:
    prompt.extend(["Query image: ", query_image.to_part(), ";\n"])
  else:
    prompt.extend(["Query image: ", query_image.to_part(), ", Zoomed out image: ",
                   query_image_with_margin.to_part(), ";\n"])

  for i, result in enumerate(matches):
    prompt.append(f"Candidate {chr(i + ord('A'))} : ")
    prompt.append(f"title - {result.product.product_title}, ")
    prompt.extend(["image - ", result.product.image.to_part(), ";\n"])
  prompt.append("</input>")
  return prompt


def create_product_recognition_prompt_margin_letter_reverse(
    query_image: GcsUri,
    query_image_with_margin: Union[GcsUri, None],
    matches: Sequence[Match],
) -> Sequence[Any] | None:
  """Creates the Product Recognition Gemini reranking prompt."""
  if query_image.object_name is None or not matches:
    return None

  prompt = ["<input>"]

  for i, result in enumerate(matches):
    prompt.append(f"Candidate {chr(i + ord('A'))} : ")
    prompt.append(f"title - {result.product.product_title}, ")
    prompt.extend(["image - ", result.product.image.to_part(), ";\n"])

  if query_image_with_margin is None:
    prompt.extend(["Query image: ", query_image.to_part(), ";\n"])
  else:
    prompt.extend(["Query image: ", query_image.to_part(), ", Zoomed out image: ",
                   query_image_with_margin.to_part(), ";\n"])

  prompt.append("</input>")

  return prompt



def expand_matches_with_front_image(matches: Sequence[Match],
                                    gtin_image_dict: dict[int, Any]):
  expanded_matches = []
  for result in matches:
    expanded_matches.append(result)
    if len(result.product.gtins) > 0:
      product_gtin = int(result.product.gtins[0])
      if product_gtin in gtin_image_dict:
        front_image = gtin_image_dict[product_gtin]
        if front_image.object_name != result.product.image.object_name:
          # Only add if retrieved image is not front image.
          new_product = dataclasses.replace(result.product, image=front_image)
          new_result = dataclasses.replace(result, product=new_product)
          expanded_matches.append(new_result)

  return expanded_matches


def create_product_title_prompt_margin_debug(
    query_image: GcsUri,
    query_image_with_margin: Union[GcsUri, None],
) -> Sequence[Any] | None:
  """Creates the Product Recognition Gemini reranking prompt."""
  if query_image.object_name is None:
    return None

  prompt = ["<input>"]
  if query_image_with_margin is None:
    prompt.extend(["Query image (white padded): ", query_image.to_part(), ";\n"])
  else:
    prompt.extend(["Query image (white padded): ", query_image.to_part(), ", Zoomed out image: ",
                   query_image_with_margin.to_part(), ";\n"])

  prompt.append("</input>")
  return prompt


def parse_gemini_response(response: str) -> dict[str, str]:
  """Parses the response from the model.

  Args:
      response (str): The response from the model.

  Returns:
      dict[str, str]: A dictionay of reponses and their file names.
  """

  if response.startswith("```"):
    lines = response.splitlines()
    json_str = ""
    json_start = False
    for line in lines:
      if line.startswith("```"):
        if not json_start:
          json_start = True
          continue
        else:
          break

      if json_start:
        json_str += line
  else:
    json_str = response
  return json.loads(json_str)

In [ ]:
#@title Read Catalog file and build a gtin to image dictionary
import pandas as pd
import requests
from PIL import Image
from io import BytesIO

catalog_csv_path=f"gs://{BUCKET}/{DATASET}"

catalog_df = pd.DataFrame()
gtin_image_dict = {}
gtin_title_dict = {}

# Loads the gtin to image lookup.
catalog_df = pd.read_csv(catalog_csv_path)
catalog_df["images"] = catalog_df["images"].apply(lambda images: images.split('||')[0])

def from_web_uri_no_exception(web_uri: str) -> GcsUri | None:
    try:
      return GcsUri.from_web_uri(web_uri, skip_download=True)
    except Exception as e:
      return

pandarallel.pandarallel.initialize(nb_workers=12, progress_bar=True)

catalog_df["gcs_uri"] = catalog_df["images"].parallel_apply(from_web_uri_no_exception)

for index, row in catalog_df.iterrows():
  gtins = row["gtins"].split("|")
  for gtin in gtins:
    gtin_title_dict[int(gtin)] = row["product_title"]
    if row["gcs_uri"]:
      gtin_image_dict[int(gtin)] = row["gcs_uri"]


print(f"Loadded {len(gtin_image_dict)} gtins")

INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


Loadded 55758 gtins

In [ ]:
# @title Prepare Vertex Matching Engine
# @markdown If `INDEX_NAME` and/or `INDEX_ENDPOINT` are provided, they will be
# @markdown reused. Otherwise, they will be created.\
# @markdown **Note**: that index deployment to an endpoint takes about 20
# @markdown minutes.

# Create the Index and IndexEndpoint.
INDEX_NAME = ""  # @param {type: "string"}
INDEX_ENDPOINT_NAME = ""  # @param {type: "string"}
DEPLOYED_INDEX_ID = ""  # @param {type: "string"}

# Reuse or create the index.
if INDEX_NAME:
  INDEX = aiplatform.MatchingEngineIndex(INDEX_NAME)
else:
  INDEX = aiplatform.MatchingEngineIndex.create_tree_ah_index(
      display_name="Product Recognition Index",
      dimensions=EMBEDDING_DIMENSION,
      approximate_neighbors_count=500,
      distance_measure_type="COSINE_DISTANCE",
      index_update_method="STREAM_UPDATE",
      leaf_node_embedding_count=10_000,
      leaf_nodes_to_search_percent=20.0,
  )

# Reuse or create and deploy the index endpoint.
if INDEX_ENDPOINT_NAME:
  INDEX_ENDPOINT = aiplatform.MatchingEngineIndexEndpoint(INDEX_ENDPOINT_NAME)
else:
  INDEX_ENDPOINT = aiplatform.MatchingEngineIndexEndpoint.create(
      display_name="Product Recognition Index Endpoint",
      public_endpoint_enabled=True,
  )
  INDEX_ENDPOINT = INDEX_ENDPOINT.deploy_index(
      index=INDEX, deployed_index_id=DEPLOYED_INDEX_ID
  )


In [ ]:
#@title [RunOnce] Load data and build index

# Load the dataset, copying images into GCS.
# Configure a limit. If rerunning the code, no need to load all data.
print("Loading Dataset...")
LIMIT = 1000000
products = load_dataset(DATASET, limit=LIMIT)

# Visualize some sample products.
print("\n\n\n\nVisualizing Products...")
print(len(products))
visualize_products(random.choices(products, k=10))

# Upsert the datapoints into the index. This will also require generating
# embeddings for the images.
print("\n\n\n\nUpserting Datapoints...")
upsert_times = upsert_dataset(products)
print(
    f"Upserted {len(upsert_times)} batches. Each batch took on average "
    f"{sum(upsert_times) / len(upsert_times)} seconds."
)

In [ ]:
#@title [Optional][RunOnce] Pad product image to square to be used in query

from PIL import Image
import pandas as pd
import io
import os


def generate_padded_image(detection_csv: str,
                          destination: str,
                          nb_workers: int = 12,
                          limit: int | None = None):
  """Loads a CSV detection dataset and create padded query images."""

  def pad_and_save(image_url: str):
    try:
      image_gcs = GcsUri.from_web_uri(image_url)
      image_bytes = image_gcs.to_bytes()

      # Create the image from the bytes
      image = Image.open(io.BytesIO(image_bytes))

      width, height = image.size

      # Determine the larger dimension
      max_dim = max(width, height)

      # Create a new square image with a white background
      # 'RGB' mode for color images, 'L' for grayscale if needed, but 'RGB' is safer for general use
      padded_image = Image.new("RGB", (max_dim, max_dim), (255, 255, 255)) # White background (R, G, B)

      # Calculate paste position to center the original image
      x_offset = (max_dim - width) // 2
      y_offset = (max_dim - height) // 2

      # Paste the original image onto the center of the new square image
      padded_image.paste(image, (x_offset, y_offset))

      # Save the image to bytes and store it in gcs.
      padded_bytes = io.BytesIO()
      padded_image.save(padded_bytes, format='JPEG', quality=95)

      save_path = os.path.join(destination, image_url.split("/")[-1])
      blob = GCS_BUCKET.blob(save_path)
      blob.upload_from_string(padded_bytes.getvalue(), "image/jpeg")
    except Exception as e:
      print(e)
      return

  pandarallel.pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)

  # Extract the dataset CSV text.
  blob = GCS_BUCKET.blob(detection_csv)
  text = blob.download_as_string().decode("utf-8")

  if detection_csv.endswith("csv"):
    # Load the CSV text into a dataframe for easier processing.
    df = pd.read_csv(io.StringIO(text), header=0)
    df.parallel_apply(lambda row: pad_and_save(row['imageUri']), axis=1)
  else: # assume to be json
    data_json = json.loads(text)
    df = pd.DataFrame(data_json)
    df.parallel_apply(lambda row: pad_and_save(row['frameUrl']), axis=1)

generate_padded_image('CSV', 'padded_cache_directory')


In [ ]:
#@title [Optional][RunOnce] Segment product from original frame with a margin to be used in query - scan 2

from PIL import Image
import pandas as pd
import io
import os


def generate_segments(detection_csv: str,
                      destination: str,
                      margin: float = 0.2,
                      nb_workers: int = 12,
                      limit: int | None = None):
  """Loads a CSV detection dataset and create segments."""
  def normalize(bbox, width, height):
    x0 = bbox[0]
    if x0 < 0:
      x0 = 0
    y0 = bbox[1]
    if y0 < 0:
      y0 = 0
    x1 = bbox[2]
    if x1 >= width:
      x1 = width - 1
    y1 = bbox[3]
    if y1 >= height:
      y1 = height - 1
    return (x0, y0, x1, y1)


  def segment_detection_and_save(frame_url: str, image_url: str,
                                 x: int, y: int, w: int, h: int):
    try:
      frame_gcs = GcsUri.from_web_uri(frame_url)
      frame_bytes = frame_gcs.to_bytes()

      # Create the image from the bytes
      image = Image.open(io.BytesIO(frame_bytes))

      # Maks the bounding box square.
      center = (x + w/2, y + h/2)
      if w < h:
        width = h * (1 + margin)
      else:
        width = w * (1 + margin)

      bbox = (center[0] - width/2, center[1] - width/2,
              center[0] + width/2, center[1] + width/2)

      cropped_image = image.crop(normalize(bbox, image.width, image.height))

      # Save the image to bytes and store it in gcs.
      cropped_bytes = io.BytesIO()
      cropped_image.save(cropped_bytes, format='JPEG', quality=95)


      save_path = os.path.join(destination, image_url.split("/")[-1])
      blob = GCS_BUCKET.blob(save_path)
      blob.upload_from_string(cropped_bytes.getvalue(), "image/jpeg")
    except Exception as e:
      print(e)
      return

  pandarallel.pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)

  # Extract the dataset CSV text.
  blob = GCS_BUCKET.blob(detection_csv)
  text = blob.download_as_string().decode("utf-8")

  # Load the CSV text into a dataframe for easier processing.
  if detection_csv.endswith("csv"):
    df = pd.read_csv(io.StringIO(text), header=0)

    df.parallel_apply(lambda row: segment_detection_and_save(
        row['frameUrl'], row['imageUri'], row['x'], row['y'], row['w'], row['h']),
                    axis=1)
  else: # assume to be json.
    data_json = json.loads(text)
    df = pd.DataFrame(data_json)
    df.parallel_apply(lambda row: segment_detection_and_save(
        row['imageUri'], row['frameUrl'], row['x'], row['y'], row['w'], row['h']),
                    axis=1)

  #df = df.dropna()
  #print(f"Total number of rows: {len(df)}")
  #return [Product.from_df_row(row) for row in df.iterrows()]

generate_segments('CSV_PATH', 'CACHE_PATH', margin=1.0)


In [ ]:
#@title [Demo] End2End flow for 1 product

cns_path = ""

gcs_uris = [GcsUri.from_parts(BUCKET,
                              f"{cache}/CACHED_URI/segments/"
                              + cns_path.split('/')[-1]
                              )]
SEGMENT_PATH = ""
PADDED_PATH = ""

# Load the queryset.
for i, image in enumerate(gcs_uris):
  print(f"\n\n\n\n\nSending query #{i + 1}...")
  start_time = time.perf_counter()

  image_with_margin = GcsUri.from_parts(
      image.bucket_name,
      os.path.join(SEGMENT_PATH, image.object_name.split('/')[-1]))

  padded_image = GcsUri.from_parts(
      image.bucket_name,
      os.path.join(PADDED_PATH, image.object_name.split('/')[-1]))

  visualize_products(padded_image, 512)
  visualize_products(image_with_margin, 512)

  # Find the nearest neighbors from Vertex Matching Engine.
  print("\n\nFinding nearest neighbors...")
  matches = padded_image.find_neighbors()

  print("\n\nVisualizing nearest neighbors...")
  visualize_products(matches)

  # Prepare and send the Gemini reranking request.
  print("\n\nPerforming Gemini reranking...")
  prompt = create_product_recognition_prompt_margin_letter_reverse(padded_image, image_with_margin, matches)

  gemini_response = client.models.generate_content(model=GEMINI_MODEL_VERSION,
                                                   contents=prompt,
                                                   config=gemini_punting_letter_config)
  gemini_response_text = gemini_response.candidates[0].content.parts[0].text
  end_time = time.perf_counter()
  print(f"Gemini response: {gemini_response_text}")
  result = parse_gemini_response(gemini_response_text)
  index = ord(result['candidate']) - ord('A')
  if index >= 0:
    visualize_products(matches[index].product)
  print(f"Total query time: {end_time - start_time} seconds")

Sending query #1...

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x0…

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x0…

Finding nearest neighbors...

Visualizing nearest neighbors...

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x0…

Performing Gemini reranking...

Gemini response: ```json
{
  "color": "red and white",
  "title": "Breakstones Sour Cream - 8 Oz",
  "candidate": "J"
}
```

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x0…

Total query time: 4.1797732349996295 seconds